# 🛟 Tuning PID del Float — tool plug-and-play

Questo notebook ti aiuta a **tarare il controllo di profondità** del Float anche se non sei esperto di controlli automatici.

**Cosa fa:**
1. Ti dà i **valori da inserire nei campi della GUI NEXUS** (`kp, ki, kd, ...`) e una stima di **`u_neutral`**.
2. **Analizza i dati** di un tuo test e ti dice **cosa correggere**, **riconoscendo da solo le fasi** del profilo: solo discesa, solo salita o profilo completo (discesa → hold → salita), con metriche e diagnosi **per fase**. Puoi:
   - **incollare** la tabella copiata dalla vista "Raw chart" della GUI (o un JSON), oppure
   - **caricare un CSV/TXT** (export GUI o log flash `DUMP_LOG` del Float).

**Come si usa:** in alto **Runtime → Esegui tutto**. Poi nella sezione 4 incolli i dati (o carichi il CSV); se non metti nulla, viene usato un log di esempio.

> ⚠️ Il **target di discesa è riferito al FONDO del float** (il barometro in cima legge `target − 0,51 m`); il **target di salita è riferito al TOP** (come il campo `ascent_target` della GUI) e il tool lo converte da solo (+0,51 m), esattamente come fa il firmware. In vasca bassa (< ~1 m) il sensore è a pochi cm dal pelo → oscillazioni **non** colpa del PID.
>
> ℹ️ Incolla i dati dalla vista **"Raw chart"** della GUI **così come sono**: righe tipo `0 → 0.53 m → 98.47 kPa → 0.42 u` (tab, unità e virgole decimali sono gestiti, con o senza intestazione; vedi `esempio_gui_dump.txt`). Il tool capisce **2, 3 o 4 colonne**. Se c'è anche la **siringa (`u`)** fa il tuning completo (`u_neutral`, saturazione); se la tabella ha solo profondità/pressione, analizza la profondità e per `u` usa il **log flash** `DUMP_LOG`.
>
> ℹ️ Se c'è la colonna **pressione**, la profondità viene **ricostruita dal dato grezzo** (riferimento FONDO per tutto il log): il log di missione del firmware cambia riferimento fondo/top al cambio fase e introduce salti fittizi di ~0,5 m che falserebbero metriche e stima di `u_neutral`. La colonna originale resta in `depth_log`.
>
> ℹ️ Le **fasi** vengono riconosciute dalla colonna `phase` del log flash (`descending`/`hold_2_5m` → discesa, `ascending`/`hold_40cm` → salita) oppure, se manca (dump GUI), dalla **forma della traiettoria** di profondità. Se vuoi forzare l'analisi su una sola fase, imposta `fase = "discesa"` o `"salita"` nella sezione 4.

In [ ]:
# @title 1) Setup — esegui questa cella per prima
import sys, subprocess, io, os, json

def _ensure(pkgs):
    for p in pkgs:
        try:
            __import__(p)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

_ensure(["numpy", "pandas", "matplotlib"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- Default e limiti presi DAL FIRMWARE (fonte di verità: include/config.h,
#      lib/runtime_config/, lib/profile/). Non inventati. ----
FW = {
    "pid_default": dict(kp=1.7, ki=0.1, kd=0.3, period_ms=50,
                        alpha_d=0.25, integral_limit=5.0,
                        min_retarget_frac=0.001, u_neutral=0.011),
    "pid_range": dict(period_ms=(20, 500), alpha_d=(0.05, 1.0)),
    "u_min": 0.0, "u_max": 0.92,
    "float_length_m": 0.51,
    "sensor_to_top_m": 0.0,   # FLOAT_TOP_TO_SENSOR_M: sensore in cima al float
    "profile_default": dict(descent_target=2.5, ascent_target=0.40,
                            depth_tolerance=0.33, hold_time=30,
                            descent_timeout=180, ascent_timeout=120,
                            surface_offset=0.10),
}
RAW_EXAMPLE_URL = ("https://raw.githubusercontent.com/PoliTOcean/Float/"
                   "master/tools/pid_tuning/esempio_gui_dump.txt")

print("Ambiente:", "Google Colab" if IN_COLAB else "locale", "| setup OK")

## 2) I tre concetti minimi

- **`u` (apertura siringa)**: da **0** (vuota → *galleggia*) a **1** (piena → *affonda*). È l'**uscita** che il controllo calcola da solo; nei log è una colonna. **Non si imposta a mano.**
- **PID**: decide `u` dall'errore di profondità. Tre manopole — **P** (reazione ora), **I** (recupero dell'errore che resta), **D** (freno/anticipo).
- **`u_neutral`**: apertura siringa di **assetto neutro** (il float né sale né scende). È un **parametro** da impostare; qui lo **stimiamo dai dati** (se è presente `u`). Diverso da `u`!


In [ ]:
# @title 2) Funzioni (caricamento dati, segmentazione fasi, analisi, stima u_neutral)

def mostra_valori(d, titolo=""):
    if titolo:
        print(titolo)
        print("-" * len(titolo))
    for k, v in d.items():
        print(f"  {k:18s} = {v}")

def _leggi_csv(src):
    if isinstance(src, str):
        if src.startswith(("http://", "https://")):
            from urllib.request import urlopen
            content = urlopen(src, timeout=15).read().decode("utf-8")
        else:
            with open(src, 'r', encoding='utf-8') as f:
                content = f.read()
    else:
        content = src.read().decode('utf-8')
    try:
        # Cerchiamo di leggere con pandas sniffer
        df = pd.read_csv(io.StringIO(content), sep=None, engine="python")
        if df.shape[1] < 2:
            raise ValueError("Troppe poche colonne, probabile separatore non standard")
        # Dump GUI salvato su file: celle con unita' ("0.53 m", "98.47 kPa") e
        # prima riga dati scambiata per header -> servono almeno 2 colonne
        # numeriche, altrimenti si riparsa il testo grezzo riga per riga.
        numeriche = sum(pd.to_numeric(df[c], errors="coerce").notna().mean() > 0.5
                        for c in df.columns)
        if numeriche < 2:
            raise ValueError("Celle non numeriche (unita' m/kPa nel testo?)")
        return df
    except Exception:
        # Fallback ultra-robusto (unita' nelle celle, tabelle salvate con spazi)
        return _da_testo(content)

def _ricostruisci_depth(df):
    """Il log di missione del firmware cambia riferimento di profondita' al
    cambio fase (FONDO in discesa/risalita, TOP durante hold_40cm): la colonna
    depth puo' avere salti fittizi di ~0,5 m che falsano metriche e stime.
    Se c'e' la pressione (kPa), ricostruiamo la profondita' dal dato grezzo,
    riferita al FONDO per tutto il log. L'originale resta in 'depth_log'."""
    if "pressure" not in df.columns or "depth" not in df.columns:
        return df
    p = pd.to_numeric(df["pressure"], errors="coerce")
    if p.isna().all():
        return df
    n_atm = max(3, len(p) // 10)
    p_atm = p.nsmallest(n_atm).median()
    depth_p = (p - p_atm) / 9.79 + FW["float_length_m"]  # kPa -> m, rif. FONDO
    scarto = (df["depth"] - depth_p).abs()
    # Sostituiamo solo con la firma del cambio riferimento: log per lo piu'
    # coerente con la pressione (mediana piccola) ma con salti localizzati.
    if scarto.median() < 0.15 and scarto.max() > 0.3:
        df["depth_log"] = df["depth"]
        df["depth"] = depth_p
        print("Nota: la profondita' del log cambiava riferimento (fondo/top) al "
              "cambio fase -> ricostruita dalla pressione, riferimento FONDO "
              f"(p_atm stimata {p_atm:.2f} kPa). Originale nella colonna 'depth_log'.")
    return df

def _normalizza(df):
    cols = list(df.columns)
    numeric_header = all(str(c).replace(".", "", 1).replace("-", "", 1).isdigit()
                         for c in cols)
    if numeric_header:
        df = df.copy()
        df.columns = list(range(df.shape[1]))
        cols = list(df.columns)

    ren, used = {}, set()
    for c in cols:
        cl = str(c).strip().lower()
        target = None
        if cl in ("t", "time", "timestamp", "times", "time_s", "tempo", "t_ms", "millis", "ms") or "time (s)" in cl:
            target = "t"
        elif "sensor" in cl and "depth" in cl:
            target = "sensor_depth"
        elif ("depth" in cl or cl.startswith("profond")) and "sensor" not in cl:
            target = "depth"  # nota: NON usare "prof" generico, matcha "profile_id"
        elif cl in ("u", "syringe_u", "apertura", "apertura_siringa_u", "u_norm", "syringe"):
            target = "u"
        elif "press" in cl:
            target = "pressure"
        elif "phase" in cl or cl == "fase":
            target = "phase"
        if target and target not in used:
            ren[c] = target
            used.add(target)
    df = df.rename(columns=ren)

    # ripiego posizionale solo se non ho riconosciuto le colonne per nome
    if "depth" not in df.columns and ("u" not in df.columns and "pressure" not in df.columns):
        n = df.shape[1]
        if n == 4:            # GUI: timestamp, depth, pressure, u
            df.columns = ["t", "depth", "pressure", "u"]
        elif n >= 8:          # DUMP_LOG firmware (8 colonne)
            df = df.rename(columns={df.columns[2]: "t", df.columns[4]: "depth",
                                    df.columns[7]: "u"})

    if "depth" not in df.columns:
        raise ValueError("Non trovo la colonna profondita. Servono almeno tempo e profondita (depth_m).")

    for c in ("t", "depth", "u", "pressure", "sensor_depth"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["depth"]).reset_index(drop=True)
    if "u" in df.columns and df["u"].abs().median() > 1.5:
        print("Nota: 'u' sembra in percentuale (0..100) -> converto in 0..1.")
        df["u"] = df["u"] / 100.0

    if "t" not in df.columns or df["t"].isna().all():
        df["t"] = np.arange(len(df)) * 0.2
    df["t"] = df["t"] - df["t"].iloc[0]
    if df["t"].max() > 3600:      # quasi certamente in millisecondi
        df["t"] = df["t"] / 1000.0
    return _ricostruisci_depth(df)

def _da_json(testo):
    """Costruisce il DataFrame dal JSON 'raw' della GUI (array paralleli) o da una
    lista di punti [{timestamp, depth, pressure}, ...]."""
    obj = json.loads(testo)
    raw = obj.get("raw", obj) if isinstance(obj, dict) else obj
    if isinstance(raw, list):
        df = pd.DataFrame(raw)
    else:
        def pick(*names):
            for n in names:
                v = raw.get(n)
                if isinstance(v, list) and len(v) > 0:
                    return v
            return None
        campi = {
            "t": pick("times", "time_s", "timestamp", "t"),
            "depth": pick("depth_m", "depth", "profondita_m", "profondita"),
            "pressure": pick("pressure_kpa", "pressure", "pressione_kpa", "pressione"),
            "sensor_depth": pick("sensor_depth_m"),
            "phase": pick("phase"),
            "u": pick("syringe_u", "syringe", "u", "u_norm", "apertura"),
        }
        campi = {k: v for k, v in campi.items() if v is not None}
        if "depth" not in campi:
            raise ValueError("JSON senza profondita (depth_m/depth).")
        n = max(len(v) for v in campi.values())
        campi = {k: v for k, v in campi.items() if len(v) == n}
        df = pd.DataFrame(campi)
    return _normalizza(df)

def _ordine_header(line):
    """Deduce l'ordine delle colonne dai nomi nell'intestazione della tabella."""
    hl = line.lower()
    coppie = [("timestamp", "t"), ("time", "t"), ("tempo", "t"),
              ("depth", "depth"), ("profond", "depth"),
              ("pressure", "pressure"), ("pressione", "pressure"), ("press", "pressure"),
              ("syringe", "u"), ("siringa", "u")]
    trovati = []
    for kw, canon in coppie:
        idx = hl.find(kw)
        if idx >= 0:
            trovati.append((idx, canon))
    trovati.sort()
    ordine, visti = [], set()
    for _, canon in trovati:
        if canon not in visti:
            visti.add(canon)
            ordine.append(canon)
    return ordine

def _da_testo(testo):
    """Parsa il TESTO della tabella copiato dalla GUI (vista 'Raw chart'): righe di
    numeri separati da spazi/virgole/tab. Robusto a unita' (m, kPa, u), header,
    'N/A', timestamp in ms, decimali con la virgola (righe tab-separated) e 2-4
    colonne. Accetta il dump GUI cosi' com'e': `0\t0.53 m\t98.47 kPa\t0.42 u`.
    Se c'e' una riga d'intestazione, l'ordine delle colonne viene dedotto dai nomi
    (Timestamp/Time (s)/Depth/Pressure/Syringe)."""
    import re
    from collections import Counter
    num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
    righe = [r for r in testo.strip().splitlines() if r.strip()]
    if not righe:
        raise ValueError("Niente da leggere: incolla la tabella copiata dalla GUI.")
    ordine_header = None
    kw = ("time", "depth", "prof", "press", "syringe", "siringa")
    if any(k in righe[0].lower() for k in kw) and len(num_re.findall(righe[0])) < 2:
        ordine_header = _ordine_header(righe[0])
        righe = righe[1:]
    rows = []
    for r in righe:
        if "\t" in r:
            # riga tab-separated (copia da tabella): una virgola fra cifre
            # e' un decimale (locale it), non un separatore di colonna
            r = re.sub(r"(?<=\d),(?=\d)", ".", r)
        nums = num_re.findall(r)
        if nums:
            rows.append([float(x) for x in nums])
    if not rows:
        raise ValueError("Nessun numero trovato: incolla la tabella (una riga per campione).")
    ncol = Counter(len(r) for r in rows).most_common(1)[0][0]
    if ncol < 2:
        raise ValueError("Servono almeno 2 colonne: tempo e profondita.")
    arr = np.array([r for r in rows if len(r) == ncol], dtype=float)
    if ordine_header and len(ordine_header) == ncol:
        nomi = ordine_header
    else:
        nomi = ["t", "depth", "pressure", "u"][:min(ncol, 4)]
        if ncol > 4:
            arr = arr[:, :4]
            nomi = ["t", "depth", "pressure", "u"]
    df = pd.DataFrame(arr[:, :len(nomi)], columns=nomi)
    return _normalizza(df)

def carica(dati=""):
    """Sorgente dati: testo incollato (tabella o JSON) > upload CSV (Colab) > esempio."""
    if dati and dati.strip():
        s = dati.strip()
        try:
            df = _da_json(s) if s[0] in "[{" else _da_testo(s)
            cols = [c for c in ("t", "depth", "u", "pressure", "phase") if c in df.columns]
            print(f"Dati incollati: {len(df)} campioni. Colonne: {cols}")
            if "u" not in df.columns:
                print("Nota: niente 'u' (apertura siringa) -> analisi solo di profondita. "
                      "Per u_neutral/saturazione usa il log flash DUMP_LOG.")
            return df
        except Exception as e:
            print("Non riesco a leggere i dati incollati:", e)
            print("Uso un CSV/esempio come ripiego.")
    return carica_log()

def carica_log():
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        if up:
            nome = list(up.keys())[0]
            print("Caricato:", nome)
            return _normalizza(_leggi_csv(io.BytesIO(up[nome])))
        print("Nessun file caricato -> uso l'esempio.")
    for src in ("esempio_gui_dump.txt", "tools/pid_tuning/esempio_gui_dump.txt", RAW_EXAMPLE_URL):
        try:
            df = _normalizza(_leggi_csv(src))
            print("Uso log di esempio:", src)
            return df
        except Exception:
            continue
    raise RuntimeError("Nessun dato disponibile (incolla i dati o carica un CSV).")

def _valida_pid(d):
    d = dict(d)
    lo, hi = FW["pid_range"]["period_ms"]
    d["period_ms"] = int(min(max(d["period_ms"], lo), hi))
    lo, hi = FW["pid_range"]["alpha_d"]
    d["alpha_d"] = round(min(max(d["alpha_d"], lo), hi), 3)
    d["u_neutral"] = max(0.0, d["u_neutral"])
    for k in ("kp", "ki", "kd"):
        d[k] = max(0.0, d[k])
    return d

def _stringa_pid(d):
    return ("PID_CONFIG_SET {kp} {ki} {kd} {period_ms} {alpha_d} "
            "{integral_limit} {min_retarget_frac} {u_neutral}").format(**d)

# ---------- Segmentazione fasi (discesa / salita) ----------

def _target_salita_fondo(target_salita_top):
    """Il target di salita della GUI (ascent_target) e' riferito al TOP del float;
    il PID e il log lavorano in riferimento FONDO. Stessa conversione del
    firmware (ProfileManager::ascentTargetBottomM)."""
    return target_salita_top + FW["float_length_m"] + FW["sensor_to_top_m"]

def _segmenta_da_phase(df):
    """Segmenta usando i nomi di fase del firmware: descending/hold_2_5m ->
    discesa, ascending/hold_40cm -> salita. I tag di servizio (phase_start,
    exit_*, deployed) non appartengono a nessuna delle due."""
    ph = df["phase"].astype(str).str.lower()
    is_dis = (ph.str.contains("descend") | ph.str.contains("hold_2")).values
    is_sal = (ph.str.contains("ascend") | ph.str.contains("hold_40")).values
    idx_dis, idx_sal = np.flatnonzero(is_dis), np.flatnonzero(is_sal)
    segs = {}
    if len(idx_dis) >= 5:
        segs["discesa"] = (int(idx_dis[0]), int(idx_dis[-1]))
    if len(idx_sal) >= 5:
        i0 = int(idx_sal[0])
        if "discesa" in segs:
            i0 = max(i0, segs["discesa"][1])
        if i0 < int(idx_sal[-1]):
            segs["salita"] = (i0, int(idx_sal[-1]))
    return segs

def _segmenta_da_profondita(df, tgt_dis, tgt_sal_b, tol):
    """Euristica senza colonna 'phase': individua il plateau piu' profondo; cio'
    che lo precede e' la discesa, se dopo si risale in modo netto e' la salita."""
    d = df["depth"].rolling(5, center=True, min_periods=1).median().values
    n = len(d)
    dmax = float(np.percentile(d, 98))
    escursione = dmax - float(np.percentile(d, 2))
    if escursione < 0.3:
        # log ~piatto: una sola quota tenuta -> e' la fase col target piu' vicino
        med = float(np.median(d))
        quale = "discesa" if abs(med - tgt_dis) <= abs(med - tgt_sal_b) else "salita"
        print(f"Log senza transizioni evidenti (quota ~{med:.2f} m) -> "
              f"lo analizzo come sola {quale}.")
        return {quale: (0, n - 1)}
    # Fine del plateau profondo: ultimo campione entro ~2*tolleranza dal massimo.
    # Banda stretta apposta: se entrasse la rampa di risalita, gonfierebbe
    # l'oscillazione misurata sulla coda della discesa (falsa diagnosi).
    soglia_plateau = dmax - max(0.15, 2 * tol)
    deep = np.flatnonzero(d >= soglia_plateau)
    i_fine_plateau = int(deep[-1])
    segs = {}
    if (dmax - d[0]) > 0.3 * escursione:
        segs["discesa"] = (0, i_fine_plateau)
    if i_fine_plateau < n - 5 and \
            (dmax - float(np.min(d[i_fine_plateau:]))) > 0.3 * escursione:
        segs["salita"] = (i_fine_plateau, n - 1)
    return segs

def segmenta_fasi(df, tgt_dis, tgt_sal_b, tol, fase="auto"):
    """Capisce da solo quali fasi contiene il log: solo discesa, solo salita o
    profilo completo. Usa la colonna 'phase' del firmware se presente,
    altrimenti la forma della traiettoria. Con fase="discesa"/"salita" forza
    TUTTO il log come quella sola fase."""
    if fase in ("discesa", "salita"):
        return {fase: (0, len(df) - 1)}
    segs = {}
    if "phase" in df.columns:
        segs = _segmenta_da_phase(df)
    if not segs:
        segs = _segmenta_da_profondita(df, tgt_dis, tgt_sal_b, tol)
    if not segs:
        print("Nota: non riconosco fasi distinte -> analizzo tutto il log come discesa.")
        segs = {"discesa": (0, len(df) - 1)}
    return segs

def _taglia_riemersione(d, i0, i1, tgt_sal_b, tol):
    """Se dopo l'hold al target di salita il log prosegue con la riemersione in
    superficie, escludiamo quel tratto finale: falserebbe errore a regime e
    oscillazione della fase di salita (non e' piu' il PID a inseguire il target)."""
    soglia = tgt_sal_b - max(2 * tol, 0.25)
    seg = d[i0:i1 + 1]
    sopra = seg < soglia               # depth minore = piu' in alto del target
    idx_dentro = np.flatnonzero(~sopra)
    if len(idx_dentro) == 0:
        return i1                      # mai vicino al target: non taglio nulla
    j = int(idx_dentro[-1])
    if (len(seg) - 1 - j) > max(5, 0.1 * len(seg)):
        return i0 + j
    return i1

# ---------- Metriche, grafici, diagnosi ----------

def _metriche(df, target, tol):
    t = df["t"].values; d = df["depth"].values
    n = len(d); start = float(d[0])
    e_ss = float(np.mean(d[int(n * 0.8):]) - target)
    overshoot = (np.max(d) - target) if target >= start else (target - np.min(d))
    span = abs(target - start) if abs(target - start) > 1e-6 else 1.0
    overshoot_pct = 100.0 * max(0.0, float(overshoot)) / span
    fuori = np.abs(d - target) > tol
    settling = float(t[np.where(fuori)[0][-1]]) if fuori.any() else 0.0
    half = d[int(n * 0.5):]
    osc_pp = float(np.max(half) - np.min(half))
    err = half - np.mean(half)
    zc = np.where(np.diff(np.sign(err)) != 0)[0]
    thalf = t[int(n * 0.5):]
    periodo = float(2 * np.mean(np.diff(thalf[zc]))) if len(zc) >= 2 else float("nan")
    if "u" in df.columns:
        u = df["u"].values
        sat = float(np.mean((u <= FW["u_min"] + 1e-3) | (u >= FW["u_max"] - 1e-3)) * 100.0)
    else:
        sat = None
    sensor_depth = float(np.median(d) - FW["float_length_m"])
    return dict(start=start, e_ss=e_ss, overshoot=float(overshoot),
                overshoot_pct=overshoot_pct, settling=settling, osc_pp=osc_pp,
                periodo=periodo, sat=sat, sensor_depth=sensor_depth)

def _grafici(df, segs, targets, tol):
    colori = {"discesa": "tab:green", "salita": "tab:blue"}
    ha_u = "u" in df.columns
    fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    ax[0].plot(df["t"], df["depth"], label="profondita (fondo float)")
    for nome, (i0, i1) in segs.items():
        t0, t1 = float(df["t"].iloc[i0]), float(df["t"].iloc[i1])
        tgt, col = targets[nome], colori.get(nome, "green")
        ax[0].hlines(tgt, t0, t1, color=col, ls="--",
                     label=f"target {nome} ({tgt:.2f} m)")
        ax[0].fill_between([t0, t1], tgt - tol, tgt + tol, color=col, alpha=0.12)
        if len(segs) > 1 and i0 > 0:
            ax[0].axvline(t0, color="gray", ls=":", alpha=0.6)
    ax[0].set_ylabel("profondita [m]"); ax[0].invert_yaxis()
    ax[0].legend(loc="best"); ax[0].grid(alpha=0.3)
    if ha_u:
        ax[1].plot(df["t"], df["u"], color="orange", label="u (apertura siringa)")
        ax[1].axhline(FW["u_max"], color="red", ls=":", label="limite 0,92")
        ax[1].axhline(FW["u_min"], color="red", ls=":")
        ax[1].set_ylim(-0.05, 1.0); ax[1].set_ylabel("u [0..1]")
    elif "pressure" in df.columns:
        ax[1].plot(df["t"], df["pressure"], color="purple", label="pressione [kPa]")
        ax[1].set_ylabel("pressione [kPa]")
    else:
        ax[1].text(0.5, 0.5, "(nessun dato u / pressione)", ha="center", va="center")
    ax[1].set_xlabel("tempo [s]"); ax[1].legend(loc="best"); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

def _stampa_metriche(m, nome):
    print(f"\n=== METRICHE - fase {nome.upper()} ===")
    print(f"  errore a regime ......... {m['e_ss']:+.3f} m")
    print(f"  overshoot ............... {m['overshoot']:.3f} m ({m['overshoot_pct']:.0f}%)")
    print(f"  tempo di assestamento ... {m['settling']:.1f} s")
    osc = f"  oscillazione residua .... {m['osc_pp']:.3f} m picco-picco"
    if m["periodo"] == m["periodo"]:
        osc += f", periodo ~{m['periodo']:.1f} s"
    print(osc)
    if m["sat"] is None:
        print("  u saturata .............. n/d (manca la colonna u)")
    else:
        print(f"  u saturata (0 o 0,92) ... {m['sat']:.0f}% del tempo")
    print(f"  prof. sensore (mediana) . {m['sensor_depth']:.2f} m")

def _consigli_fase(m, tol, base):
    """Applica le regole di correzione PID alle metriche di UNA fase."""
    cons, note = dict(base), []
    if m["sat"] is not None and m["sat"] > 30:
        note.append("ATTENZIONE: la siringa resta spesso a fondo corsa (u a 0 o 0,92): e un problema di "
                    "ASSETTO/ZAVORRA o di u_neutral, non dei guadagni. Sistema prima quello.")
    grossa_osc = (m["osc_pp"] > 2 * tol) and (m["periodo"] == m["periodo"])
    if grossa_osc:
        cons["kp"] = round(cons["kp"] * 0.7, 3)
        cons["kd"] = round(cons["kd"] * 1.5, 3)
        note.append("Oscillazione ampia e regolare -> riduci kp e aumenta kd.")
    if m["overshoot_pct"] > 20 and not grossa_osc:
        cons["kd"] = round(cons["kd"] * 1.5, 3)
        if m["overshoot_pct"] > 50:
            cons["kp"] = round(cons["kp"] * 0.8, 3)
        note.append("Overshoot marcato -> aumenta kd (ed eventualmente abbassa kp).")
    sat_ok = (m["sat"] is None) or (m["sat"] < 30)
    if abs(m["e_ss"]) > max(tol, 0.03) and sat_ok:
        cons["ki"] = round(cons["ki"] * 1.8, 3)
        note.append(f"Errore a regime {m['e_ss']:+.2f} m -> aumenta ki per recuperarlo.")
    if (m["osc_pp"] < tol and abs(m["e_ss"]) < tol and m["overshoot_pct"] < 20):
        note.append("OK: risposta gia buona in questa fase.")
    return cons, note

def _unisci_consigli(consigli_fase, base):
    """Il firmware usa UN solo set PID per discesa e salita: fra i consigli delle
    due fasi prendiamo il kp piu' prudente (min) e kd/ki piu' incisivi (max)."""
    cons = dict(base)
    cons["kp"] = min(c["kp"] for c in consigli_fase.values())
    cons["kd"] = max(c["kd"] for c in consigli_fase.values())
    cons["ki"] = max(c["ki"] for c in consigli_fase.values())
    return cons

def analizza(df, target_discesa_m=None, target_salita_m=None,
             tolleranza_m=0.10, attuali=None, fase="auto"):
    """Analizza il log riconoscendo da solo le fasi presenti (discesa, salita o
    entrambe). target_discesa_m e' riferito al FONDO (campo GUI descent_target),
    target_salita_m al TOP (campo GUI ascent_target, convertito internamente).
    fase: "auto" | "discesa" | "salita" (forza tutto il log come quella fase)."""
    base = dict(FW["pid_default"])
    if attuali:
        base.update({k: v for k, v in attuali.items() if v is not None})
    if target_discesa_m is None:
        target_discesa_m = FW["profile_default"]["descent_target"]
    if target_salita_m is None:
        target_salita_m = FW["profile_default"]["ascent_target"]
    tgt_sal_b = _target_salita_fondo(target_salita_m)
    targets = {"discesa": float(target_discesa_m), "salita": float(tgt_sal_b)}

    segs = segmenta_fasi(df, targets["discesa"], targets["salita"],
                         tolleranza_m, fase)
    if "salita" in segs:
        i0, i1 = segs["salita"]
        i1n = _taglia_riemersione(df["depth"].values, i0, i1,
                                  targets["salita"], tolleranza_m)
        if i1n < i1:
            print("Nota: escludo dall'analisi della salita il tratto finale di "
                  "riemersione in superficie (non e' piu' il PID a inseguire il target).")
            segs["salita"] = (i0, i1n)
    ordine = sorted(segs, key=lambda k: segs[k][0])
    print("Fasi riconosciute: " + "; ".join(
        f"{nome} t={df['t'].iloc[segs[nome][0]]:.0f}-{df['t'].iloc[segs[nome][1]]:.0f} s "
        f"(target {targets[nome]:.2f} m rif. FONDO)" for nome in ordine))
    if "salita" in segs:
        print(f"  (target salita GUI {target_salita_m:.2f} m rif. TOP -> "
              f"{targets['salita']:.2f} m rif. FONDO, come nel firmware)")

    _grafici(df, segs, targets, tolleranza_m)

    metriche, consigli, note_fasi = {}, {}, {}
    for nome in ordine:
        i0, i1 = segs[nome]
        seg = df.iloc[i0:i1 + 1].copy()
        seg["t"] = seg["t"] - seg["t"].iloc[0]
        if len(seg) < 10:
            print(f"\nFase {nome}: troppi pochi campioni ({len(seg)}), la salto.")
            continue
        m = _metriche(seg, targets[nome], tolleranza_m)
        metriche[nome] = m
        _stampa_metriche(m, nome)
        consigli[nome], note_fasi[nome] = _consigli_fase(m, tolleranza_m, base)

    print("\n=== DIAGNOSI E CONSIGLI ===")
    note = []
    if "u" not in df.columns:
        note.append("MANCA la colonna 'u' (apertura siringa): NON posso stimare u_neutral ne la "
                    "saturazione. Per il tuning completo usa il log flash (DUMP_LOG, che include "
                    "'syringe_u') oppure aggiungi syringe_u all'export della GUI.")
    if "phase" in df.columns and df["phase"].astype(str).str.contains("emergency", case=False, regex=False).any():
        note.append("ATTENZIONE: nel log compare un EMERGENCY STOP (sicurezza TOF): il profilo si e "
                    "interrotto per sicurezza, non e un problema di tuning. Controlla hardware/assetto.")
    m_fondale = metriche.get("discesa") or next(iter(metriche.values()), None)
    if m_fondale and m_fondale["sensor_depth"] < 0.15:
        note.append("ATTENZIONE: barometro a < 15 cm dal pelo -> test poco affidabile, serve una vasca "
                    "piu profonda. L'oscillazione qui NON e colpa del PID.")
    for nn in note:
        print("  - " + nn)
    for nome in ordine:
        for nn in note_fasi.get(nome, []):
            print(f"  - [{nome}] " + nn)

    if not consigli:
        print("Nessuna fase analizzabile: controlla i dati.")
        return dict(fasi=segs, metriche=metriche, consigliati=None)
    if len(consigli) > 1:
        cons = _unisci_consigli(consigli, base)
        print("\nNota: il firmware usa UN solo set PID per entrambe le fasi -> "
              "combino i consigli (kp piu' prudente, kd/ki piu' incisivi).")
    else:
        cons = next(iter(consigli.values()))
    cons = _valida_pid(cons)
    print("\n=== VALORI PID CONSIGLIATI (mettili nella GUI) ===")
    mostra_valori(cons)
    print("\nStringa firmware equivalente:")
    print("  " + _stringa_pid(cons))
    return dict(fasi=segs, metriche=metriche, consigliati=cons)

def stima_u_neutral(df, targets, tolleranza_m=0.10):
    """Stima u_neutral dai campioni stabili vicino a una delle quote in `targets`
    (lista di profondita' rif. FONDO, es. [target discesa, target salita+0,51])."""
    if "u" not in df.columns:
        print("Impossibile stimare u_neutral: manca la colonna 'u' (apertura siringa).")
        print("-> Usa il log flash (DUMP_LOG) che include 'syringe_u', "
              "oppure aggiungi syringe_u all'export della GUI.")
        return None
    targets = np.atleast_1d(np.asarray(targets, dtype=float))
    t = df["t"].values; d = df["depth"].values; u = df["u"].values
    vel = np.gradient(d, t)
    # Solo campioni SOMMERSI e ~fermi: a galla la spinta extra del volume
    # emerso rende u scorrelata dall'assetto neutro in quota (il float puo'
    # restare in superficie anche con la siringa quasi piena).
    sommerso = (d - FW["float_length_m"]) > 0.15
    fermo = np.abs(vel) < 0.05
    vicino = np.zeros(len(d), bool)
    for tgt in targets:
        vicino |= np.abs(d - tgt) < max(tolleranza_m, 0.05)
    mask = sommerso & fermo & vicino
    if mask.sum() < 5 and (sommerso & fermo).sum() >= 2:
        mask = sommerso & fermo
        print("Pochi punti stabili ai target: stimo dai tratti fermi in quota.")
    if mask.sum() < 2:
        k = int(len(d) * 0.8)
        mask = np.zeros(len(d), bool); mask[k:] = True
        print("Pochi punti stabili sommersi: stima dall'ultimo tratto del log "
              "(ATTENZIONE: se li' il float era a galla la stima NON vale).")
    u_neu = float(np.clip(np.median(u[mask]), FW["u_min"], FW["u_max"]))
    print(f"u_neutral stimato ~ {u_neu:.3f}  (su {int(mask.sum())} campioni stabili)")
    print("-> Mettilo come 'u_neutral' nella GUI.")
    return u_neu

print("Funzioni pronte.")

## 3) Valori di partenza consigliati

Se non hai ancora dati, **parti da questi** (sono i valori già tarati nel firmware). Inseriscili nei campi della GUI e fai un primo test.


In [ ]:
mostra_valori(FW["pid_default"], "PID - valori di partenza (mettili nella GUI)")
print()
mostra_valori(FW["profile_default"], "Profilo - valori di partenza")


## 4) Analizza il tuo test

**Due modi per dare i dati:**
- Incolla nella variabile DATI qui sotto la tabella copiata dalla vista Raw chart della GUI.
- Lascia DATI vuoto: su Colab appare il bottone per caricare un CSV, senza nulla usa l esempio.

**Fasi del profilo:** con `fase = "auto"` il tool capisce **da solo** se il log contiene solo la discesa, solo la salita o il profilo completo (discesa → hold → salita), usando la colonna `phase` se c'è o la forma della traiettoria. Se vuoi forzare l'analisi su una sola fase scegli `discesa` o `salita`. L'eventuale riemersione finale in superficie viene esclusa automaticamente dall'analisi della salita.

**Target:** `target_discesa_m` è riferito al **FONDO** del float (campo GUI `descent_target`); `target_salita_m` è riferito al **TOP** (campo GUI `ascent_target`) e viene convertito da solo (+0,51 m). Se il log copre entrambe le fasi ottieni metriche e diagnosi **per fase** e un **unico set PID consigliato** (il firmware usa gli stessi guadagni in discesa e salita).

In [ ]:
# @title 4) Carica i dati e analizza
# >>> Per usare i TUOI dati: incolla qui la TABELLA copiata dalla vista "Raw chart"
#     (INTESTAZIONE COMPRESA) oppure un JSON, fra le triple virgolette. Vuoto = CSV/esempio.
DATI = r"""

"""

# Quali fasi analizzare: "auto" riconosce da solo se il log contiene la sola
# discesa, la sola salita o il profilo completo; "discesa"/"salita" forzano
# TUTTO il log come quella sola fase.
fase = "auto"  # @param ["auto", "discesa", "salita"]
# target di discesa: riferito al FONDO del float (campo GUI descent_target)
target_discesa_m = 2.5  # @param {type:"number"}
# target di salita: riferito al TOP del float (campo GUI ascent_target);
# il tool lo converte da solo in riferimento FONDO (+0,51 m)
target_salita_m = 0.40  # @param {type:"number"}
tolleranza_m = 0.10     # @param {type:"number"}
# (facoltativo) i parametri usati in QUESTO test, per consigli relativi:
kp_attuale = 1.7   # @param {type:"number"}
ki_attuale = 0.1   # @param {type:"number"}
kd_attuale = 0.3   # @param {type:"number"}
u_neutral_attuale = 0.011 # @param {type:"number"}

df = carica(DATI)
# Calcola il nuovo u_neutral dai dati se disponibile (campioni stabili
# vicino a uno dei due target, in riferimento FONDO)
u_neutral_stimato = stima_u_neutral(
    df, [target_discesa_m, _target_salita_fondo(target_salita_m)], tolleranza_m)
if u_neutral_stimato is not None:
    u_neutral_attuale = u_neutral_stimato

risultato = analizza(df, target_discesa_m, target_salita_m, tolleranza_m,
                     attuali=dict(kp=kp_attuale, ki=ki_attuale, kd=kd_attuale,
                                  u_neutral=u_neutral_attuale),
                     fase=fase)

## 5) Simulatore didattico (facoltativo)

Per **capire** l'effetto di `kp/ki/kd` senza il Float in acqua. Cambia i valori e riesegui la cella.

> ⚠️ **Modello APPROSSIMATO**: serve solo a farsi un'idea. I valori finali vanno **sempre** validati sui dati reali.

In [ ]:
# @title Simulatore APPROSSIMATO - opzionale
sim_kp     = 1.7   # @param {type:"number"}
sim_ki     = 0.1   # @param {type:"number"}
sim_kd     = 0.3   # @param {type:"number"}
sim_target = 2.5   # @param {type:"number"}

def simula(kp, ki, kd, target, u_neutral=0.12, T=80.0, dt=0.05):
    # modello 1-DOF APPROSSIMATO: m*z'' = k_b*(u - u_neutral) - c*z'
    m, k_b, c = 6.0, 8.0, 9.0
    z, v = 0.51, 0.0
    integ, dfilt, last_e = 0.0, 0.0, None
    ts, zs, us = [], [], []
    for i in range(int(T / dt)):
        e = target - z
        integ = float(np.clip(integ + e * dt, -5.0, 5.0))
        deriv = 0.0 if last_e is None else (e - last_e) / dt
        dfilt = 0.25 * deriv + 0.75 * dfilt
        last_e = e
        u = float(np.clip(u_neutral + kp * e + ki * integ + kd * dfilt, 0.0, 0.92))
        a = (k_b * (u - u_neutral) - c * v) / m
        v += a * dt; z += v * dt
        ts.append(i * dt); zs.append(z); us.append(u)
    return np.array(ts), np.array(zs), np.array(us)

ts, zs, us = simula(sim_kp, sim_ki, sim_kd, sim_target)
fig, ax = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax[0].plot(ts, zs); ax[0].axhline(sim_target, color="green", ls="--", label="target")
ax[0].set_ylabel("profondita [m]"); ax[0].invert_yaxis()
ax[0].set_title("Simulatore APPROSSIMATO - solo per capire l'effetto dei guadagni")
ax[0].legend(loc="best"); ax[0].grid(alpha=0.3)
ax[1].plot(ts, us, color="orange"); ax[1].set_ylim(-0.05, 1.0)
ax[1].set_ylabel("u"); ax[1].set_xlabel("tempo [s]"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Modello semplificato: per le decisioni finali usa i dati reali.")
